# Pandas Foundations — Cleaning a Real (Messy) Dataset
### Data Cleaning Challenge / Customer Sales Analysis
**Goal:** Build a deliberately messy sales dataset, diagnose its problems, and clean it properly using Pandas — while comparing `.loc` vs `.iloc`, boolean filtering, `groupby`, and vectorized ops vs `.apply()`.

In [1]:
import pandas as pd
import numpy as np

pd.__version__, np.__version__

('2.3.3', '2.2.6')

## 1. Build a messy dataset, on purpose

Before cleaning anything, we need a dataset with **known, deliberate problems** so we can prove — step by step — that we actually fixed what we set out to fix.

This sales table intentionally includes:
- Missing values scattered across `amount`, `customer_age`, and `category` (not all in one column)
- The category `"Electronics"` written two inconsistent ways: `"Electronics"` and `"electronics"`

In [2]:
data = {
    'date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
              '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10'],
    'category': ['Electronics', 'Clothing', 'electronics', 'Books',
                 'Clothing', np.nan, 'Electronics', 'Books',
                 'electronics', 'Clothing'],
    'amount': [250.0, 45.5, np.nan, 30.0,
               60.0, 120.0, 300.0, np.nan,
               275.0, 55.0],
    'customer_age': [28, np.nan, 35, 42,
                      np.nan, 31, 29, 50,
                      38, np.nan]
}

df = pd.DataFrame(data)
df

,date,category,amount,customer_age
0,2024-01-01,Electronics,250.0,28.0
1,2024-01-02,Clothing,45.5,NaN
2,2024-01-03,electronics,NaN,35.0
3,2024-01-04,Books,30.0,42.0
4,2024-01-05,Clothing,60.0,NaN
5,2024-01-06,NaN,120.0,31.0
6,2024-01-07,Electronics,300.0,29.0
7,2024-01-08,Books,NaN,50.0
8,2024-01-09,electronics,275.0,38.0
9,2024-01-10,Clothing,55.0,NaN


## 2. Diagnose before touching anything

Before fixing anything, run the standard first-look tools: `.head()`, `.info()`, `.describe()`, and `.isna().sum()`. The goal here is to **state exactly what's wrong** — not fix it yet.

In [3]:
df.head()

,date,category,amount,customer_age
0,2024-01-01,Electronics,250.0,28.0
1,2024-01-02,Clothing,45.5,NaN
2,2024-01-03,electronics,NaN,35.0
3,2024-01-04,Books,30.0,42.0
4,2024-01-05,Clothing,60.0,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          10 non-null     object 
 1   category      9 non-null      object 
 2   amount        8 non-null      float64
 3   customer_age  7 non-null      float64
dtypes: float64(2), object(2)
memory usage: 448.0+ bytes


In [5]:
df.describe()

,amount,customer_age
count,8.000000,7.000000
mean,141.937500,36.142857
std,113.986352,7.904188
min,30.000000,28.000000
25%,52.625000,30.000000
50%,90.000000,35.000000
75%,256.250000,40.000000
max,300.000000,50.000000


In [6]:
df.isna().sum()

date            0
category        1
amount          2
customer_age    3
dtype: int64

### Findings

- **`.info()`** shows `category`, `amount`, and `customer_age` each have fewer non-null entries than the total 10 rows — confirming missing data in all three columns, while `date` has none.
- **`.isna().sum()`** gives the exact count: `category` → 1 missing, `amount` → 2 missing, `customer_age` → 3 missing.
- **`.describe()`** only summarizes the numeric columns (`amount`, `customer_age`) — its count row is lower than 10, matching the missing values found above.
- **`.head()`** doesn't reveal the casing issue in `category` since only the first 5 rows are shown, and the mismatch (`"Electronics"` vs `"electronics"`) isn't visible until we specifically inspect value frequencies — that comes later with `.value_counts()`.

## 3. Missing-data strategy — decided per column, not in bulk

Each column with missing values gets its own decision. A blanket "drop everything with a gap" isn't acceptable — the right choice depends on what the column means and how much data we'd lose.

**`category` (1 missing value) → `fillna()` with `"Unknown"`**

Only 1 row out of 10 is missing here — dropping it would lose an entire transaction's amount and age data just because the category label is absent. Since `category` is a label, not a number, there's no "average" to fall back on, so we fill with an explicit placeholder (`"Unknown"`) rather than guessing a real category. This keeps the row and makes the gap visible instead of silently guessing wrong.

In [7]:
df['category'] = df['category'].fillna('Unknown')
df['category']

0    Electronics
1       Clothing
2    electronics
3          Books
4       Clothing
5        Unknown
6    Electronics
7          Books
8    electronics
9       Clothing
Name: category, dtype: object

**`amount` (2 missing values) → `fillna()` with the column median**

`amount` is the core numeric value we'll later `groupby` and sum — dropping 2 of 10 rows (20% of the data) is a real loss for such a small dataset. We use the **median** rather than the mean because `amount` has a wide spread (from ~30 to 300), and the median is less skewed by the high-value outliers like the 300 transaction. This is still an assumption — recorded here explicitly, not buried in code.

In [8]:
median_amount = df['amount'].median()
df['amount'] = df['amount'].fillna(median_amount)
median_amount, df['amount']

(np.float64(90.0),
 0    250.0
 1     45.5
 2     90.0
 3     30.0
 4     60.0
 5    120.0
 6    300.0
 7     90.0
 8    275.0
 9     55.0
 Name: amount, dtype: float64)

**`customer_age` (3 missing values) → `fillna()` with the column mean**

3 of 10 rows missing is too much to drop without significantly shrinking the dataset. Age tends to cluster around a typical value without extreme outliers here (all ages are 28–50), so the **mean** is a reasonable stand-in, unlike `amount` which had a wider spread. As with `amount`, this is an assumption about what the missing ages likely were, not a certainty.

In [9]:
mean_age = df['customer_age'].mean()
df['customer_age'] = df['customer_age'].fillna(mean_age)
mean_age, df['customer_age']

(np.float64(36.142857142857146),
 0    28.000000
 1    36.142857
 2    35.000000
 3    42.000000
 4    36.142857
 5    31.000000
 6    29.000000
 7    50.000000
 8    38.000000
 9    36.142857
 Name: customer_age, dtype: float64)

In [10]:
df.isna().sum()

date            0
category        0
amount          0
customer_age    0
dtype: int64

## 4. `.loc` vs `.iloc` — proven, not assumed

`.loc[]` selects by **label** (the index value), `.iloc[]` selects by **integer position**. On a freshly-built DataFrame with a default `0, 1, 2, ...` index, label and position happen to be the same number — which is exactly why this mistake is so easy to make. The moment the DataFrame is sorted or filtered, the index no longer matches row position, and `.loc` / `.iloc` can silently point at different rows.

In [11]:
# On the original (unsorted) DataFrame, label 3 and position 3 are the same row
row_by_label = df.loc[3]
row_by_position = df.iloc[3]

row_by_label.equals(row_by_position)

True

On the original DataFrame, `df.loc[3]` and `df.iloc[3]` return the **same row**, because the index hasn't been touched yet — label `3` and position `3` happen to coincide.

In [12]:
df_sorted = df.sort_values('amount').reset_index(drop=False)
df_sorted

,index,date,category,amount,customer_age
0,3,2024-01-04,Books,30.0,42.000000
1,1,2024-01-02,Clothing,45.5,36.142857
2,9,2024-01-10,Clothing,55.0,36.142857
3,4,2024-01-05,Clothing,60.0,36.142857
4,7,2024-01-08,Books,90.0,50.000000
5,2,2024-01-03,electronics,90.0,35.000000
6,5,2024-01-06,Unknown,120.0,31.000000
7,0,2024-01-01,Electronics,250.0,28.000000
8,8,2024-01-09,electronics,275.0,38.000000
9,6,2024-01-07,Electronics,300.0,29.000000


In [13]:
df_sorted_keep_index = df.sort_values('amount')
df_sorted_keep_index

,date,category,amount,customer_age
3,2024-01-04,Books,30.0,42.000000
1,2024-01-02,Clothing,45.5,36.142857
9,2024-01-10,Clothing,55.0,36.142857
4,2024-01-05,Clothing,60.0,36.142857
7,2024-01-08,Books,90.0,50.000000
2,2024-01-03,electronics,90.0,35.000000
5,2024-01-06,Unknown,120.0,31.000000
0,2024-01-01,Electronics,250.0,28.000000
8,2024-01-09,electronics,275.0,38.000000
6,2024-01-07,Electronics,300.0,29.000000


In [14]:
label_3_sorted = df_sorted_keep_index.loc[3]
position_3_sorted = df_sorted_keep_index.iloc[3]

print(label_3_sorted)
print("---")
print(position_3_sorted)

label_3_sorted.equals(position_3_sorted)

date            2024-01-04
category             Books
amount                30.0
customer_age          42.0
Name: 3, dtype: object
---
date            2024-01-05
category          Clothing
amount                60.0
customer_age     36.142857
Name: 4, dtype: object


False

After sorting, `df_sorted_keep_index.loc[3]` still returns the row that **originally had index label 3** (the same row as before, wherever it now sits) — but `df_sorted_keep_index.iloc[3]` returns whatever row is now in the **4th position** (position 3, zero-indexed) after sorting. These are no longer the same row.

This is exactly the trap: on an unsorted DataFrame the two look interchangeable, but after any `sort_values()`, `filter`, or `drop`, `.loc` and `.iloc` can point at completely different data — silently, with no error.

## 5. Boolean filtering — `&`/`|`, and breaking `and`/`or` on purpose

Filtering a DataFrame with a condition works exactly like yesterday's NumPy boolean masking: build a True/False Series, then use it to index the DataFrame. Multiple conditions combine with `&` (and) / `|` (or) — **not** the plain Python keywords `and`/`or`, which don't work element-wise on a Series.

In [15]:
filtered_correct = df[(df['amount'] > 100) & (df['category'] == 'Electronics')]
filtered_correct

,date,category,amount,customer_age
0,2024-01-01,Electronics,250.0,28.0
6,2024-01-07,Electronics,300.0,29.0


Now, deliberately rewrite the same filter using plain `and`/`or` instead of `&`/`|`, and run it to see what actually happens.

In [16]:
try:
    filtered_broken = df[(df['amount'] > 100) and (df['category'] == 'Electronics')]
    filtered_broken
except ValueError as e:
    print("ValueError raised:")
    print(e)

ValueError raised:
The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


### Why `and`/`or` fail here

Python's `and`/`or` are built to work on a **single** True/False value at a time. But `(df['amount'] > 100)` is not one True/False — it's an entire **Series** of True/False values, one per row. When `and` tries to evaluate two Series as if they were single booleans, Pandas raises:

`ValueError: The truth value of a Series is ambiguous...`

because it has no way to collapse a whole column of True/False values into one single True or False. `&` and `|`, by contrast, are **element-wise** operators — they compare the two Series row-by-row and produce a new True/False Series, which is exactly what filtering needs. The extra parentheses around each condition are required too, because `&`/`|` have higher operator precedence than comparison operators like `>` and `==`, so without them Python groups the operators in the wrong order.

## 6. Fix the inconsistent category

`value_counts()` counts how often each distinct value appears in a column — often the fastest way to spot a casing/duplicate-category problem that isn't obvious from `.head()` alone.

In [17]:
df['category'].value_counts()

category
Clothing       3
Electronics    2
electronics    2
Books          2
Unknown        1
Name: count, dtype: int64

The fix is a single vectorized string operation applied to the entire column at once — never hand-editing individual rows. `.str.lower()` normalizes casing across every value in one pass, then `.str.capitalize()` gives it a consistent display form.

In [18]:
df['category'] = df['category'].str.lower().str.capitalize()
df['category'].value_counts()

category
Electronics    4
Clothing       3
Books          2
Unknown        1
Name: count, dtype: int64

In [19]:
df

,date,category,amount,customer_age
0,2024-01-01,Electronics,250.0,28.000000
1,2024-01-02,Clothing,45.5,36.142857
2,2024-01-03,Electronics,90.0,35.000000
3,2024-01-04,Books,30.0,42.000000
4,2024-01-05,Clothing,60.0,36.142857
5,2024-01-06,Unknown,120.0,31.000000
6,2024-01-07,Electronics,300.0,29.000000
7,2024-01-08,Books,90.0,50.000000
8,2024-01-09,Electronics,275.0,38.000000
9,2024-01-10,Clothing,55.0,36.142857


## 7. `groupby` — split, apply, combine

`groupby` splits the data into groups by a column's distinct values, computes an aggregate independently within each group, and combines the results into one answer. This turns a manual filter-and-loop exercise into one line.

In [20]:
mean_amount_per_category = df.groupby('category')['amount'].mean()
mean_amount_per_category

category
Books           60.00
Clothing        53.50
Electronics    228.75
Unknown        120.00
Name: amount, dtype: float64

In [21]:
count_per_category = df.groupby('category')['amount'].count()
count_per_category

category
Books          2
Clothing       3
Electronics    4
Unknown        1
Name: amount, dtype: int64

In [22]:
total_amount_per_category = df.groupby('category')['amount'].sum().sort_values(ascending=False)
total_amount_per_category

category
Electronics    915.0
Clothing       160.5
Books          120.0
Unknown        120.0
Name: amount, dtype: float64

### Finding

Sorting `groupby('category')['amount'].sum()` in descending order shows **Electronics** as the top category by total sales amount, driven by a small number of high-value transactions (e.g. the 250 and 300 amount rows) compared to the smaller, more frequent Clothing and Books transactions.

## 8. Vectorized vs `.apply()` — timed

A vectorized column expression runs as a compiled loop under the hood. `.apply()` with a Python function falls back to calling that function once per row — giving up the speed advantage a real array-backed structure exists to provide. Here we prove the difference on a larger synthetic dataset (built by repeating our sample data many times).

In [23]:
# Repeat our small cleaned dataset many times to get a few hundred thousand rows
big_df = pd.concat([df] * 30000, ignore_index=True)
big_df.shape

(300000, 4)

We'll add the same computed column two ways: `total_value = amount * customer_age` (an arbitrary but simple example — the point is comparing *how* it's computed, not what it means).

- **Vectorized:** `big_df['amount'] * big_df['customer_age']`
- **`.apply()`:** `big_df.apply(lambda row: row['amount'] * row['customer_age'], axis=1)`

In [24]:
%timeit big_df['total_value_vectorized'] = big_df['amount'] * big_df['customer_age']

1.73 ms ± 85.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [25]:
%timeit big_df['total_value_apply'] = big_df.apply(lambda row: row['amount'] * row['customer_age'], axis=1)

6.4 s ± 197 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
big_df['amount_x_age_vectorized'] = big_df['amount'] * big_df['customer_age']
big_df['amount_x_age_apply'] = big_df.apply(lambda row: row['amount'] * row['customer_age'], axis=1)

(big_df['amount_x_age_vectorized'] == big_df['amount_x_age_apply']).all()

np.True_

### Recorded timing results

| Method | Time (per loop, from `%timeit`) |
|---|---|
| Vectorized (`amount * customer_age`) | *1.73 ms ± 85.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)* |
| `.apply(lambda row: ..., axis=1)` | *6.4 s ± 197 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)* |

The vectorized version is dramatically faster — typically **10x–100x**, because it runs as a single compiled NumPy operation over the whole column at once, while `.apply(..., axis=1)` calls a Python function separately for every single row, which carries real per-call overhead multiplied by hundreds of thousands of rows.

## Summary

Starting from a deliberately messy 10-row sales dataset, this notebook:

- Diagnosed missing data across `category`, `amount`, and `customer_age` using `.info()` and `.isna().sum()`
- Filled each column's gaps with a strategy justified individually — `"Unknown"` for category, median for `amount`, mean for `customer_age`
- Proved `.loc` and `.iloc` diverge after sorting, even though they matched on the original index
- Reproduced the `and`/`or` vs `&`/`|` failure on a Series, and confirmed why the vectorized operators are required
- Fixed a real casing inconsistency (`"Electronics"` / `"electronics"`) in one vectorized string operation
- Used `groupby` to find mean amount, transaction count, and the top category by total sales
- Timed vectorized column math against `.apply()` on a ~300k-row dataset, confirming the vectorized version is dramatically faster while producing identical results